# Avaliação NER — Silver vs. Gold (SIREVA-SUS)

Compara a anotação silver (GPT-4o-mini) com o gold standard (revisão manual).

**Input:** `data/silver/*.jsonl` + `data/gold/gold_sample.jsonl` (180 sentenças)  
**Output:** métricas seqeval, matriz de confusão, análise de erros

## 1. Setup

In [ ]:
import json
import sys
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

try:
    from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
    from seqeval.scheme import IOB2
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'seqeval'])
    from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
    from seqeval.scheme import IOB2

sys.path.insert(0, str(Path('.').resolve() / 'src'))
from config import DATA_SILVER, DATA_GOLD, ENTITY_TYPES

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11

FIGURES_DIR = Path('.').resolve() / 'data' / 'stats' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENTITY_NAMES = [e['name'] for e in ENTITY_TYPES]
ENTITY_COLORS = ['#8e44ad', '#9b59b6', '#16a085', '#2980b9',
                 '#7f8c8d', '#27ae60', '#d35400', '#e74c3c']

def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

print('Setup OK')

## 2. Carregar dados

In [ ]:
# Gold
gold_recs = read_jsonl(DATA_GOLD / 'gold_sample.jsonl')
gold_by_id = {r['sentenca_id']: r for r in gold_recs}

# Silver — apenas sentenças presentes no gold
silver_by_id = {}
for sf in sorted(DATA_SILVER.glob('*.jsonl')):
    for r in read_jsonl(sf):
        if r['sentenca_id'] in gold_by_id:
            silver_by_id[r['sentenca_id']] = r

print(f'Gold:   {len(gold_recs)} sentenças')
print(f'Silver: {len(silver_by_id)} correspondências')

## 2.1 Correções na anotação silver

Mesmas correções aplicadas no `Corpus_Statistics.ipynb`:
- Remove SOROTIPO falso-positivo (palavra "sorotipo" como cabeçalho de coluna)
- Normaliza FAIXA_ETARIA (`≥ 5 anos` → `≥5 anos`, `< 5 anos` → `<5 anos`)

In [ ]:
_FA_NORM = re.compile(r'([<>≥≤])\s+')

def fix_silver(r):
    new_tags = list(r['bio_tags'])
    new_spans = []
    for sp in r.get('spans', []):
        if sp['tipo'] == 'SOROTIPO' and sp['texto'].strip().lower() == 'sorotipo':
            for i, (tok, tag) in enumerate(zip(r['tokens'], new_tags)):
                if tok.lower() == 'sorotipo' and tag == 'B-SOROTIPO':
                    new_tags[i] = 'O'
        else:
            if sp['tipo'] == 'FAIXA_ETARIA':
                sp = {**sp, 'texto': _FA_NORM.sub(r'\1', sp['texto'])}
            new_spans.append(sp)
    r['bio_tags'] = new_tags
    r['spans'] = new_spans
    return r

silver_by_id = {sid: fix_silver(r) for sid, r in silver_by_id.items()}
print('Correções aplicadas ao silver.')

## 3. Preparar sequências (seqeval)

In [ ]:
# Ordena pelo sentenca_id para garantir alinhamento
ordered_ids = sorted(gold_by_id.keys())

y_true, y_pred = [], []
for sid in ordered_ids:
    g = gold_by_id[sid]
    s = silver_by_id[sid]
    assert g['tokens'] == s['tokens'], f'Token mismatch: {sid}'
    y_true.append(g['bio_tags'])
    y_pred.append(s['bio_tags'])

print(f'Sentenças avaliadas: {len(y_true)}')
print(f'Tokens totais:       {sum(len(t) for t in y_true):,}')
print(f'Entidades gold:      {sum(t.count("B-") for seq in y_true for t in seq if t.startswith("B-")):,}')
n_gold_ents = sum(1 for seq in y_true for t in seq if t.startswith('B-'))
n_pred_ents = sum(1 for seq in y_pred for t in seq if t.startswith('B-'))
print(f'Entidades gold:      {n_gold_ents:,}')
print(f'Entidades silver:    {n_pred_ents:,}')

## 4. Métricas globais (seqeval)

In [ ]:
p  = precision_score(y_true, y_pred, scheme=IOB2, average='micro')
r  = recall_score(y_true, y_pred, scheme=IOB2, average='micro')
f1 = f1_score(y_true, y_pred, scheme=IOB2, average='micro')

print('=' * 40)
print('  AVALIAÇÃO SILVER vs. GOLD')
print('=' * 40)
print(f'  Precision (micro): {p:.3f}')
print(f'  Recall    (micro): {r:.3f}')
print(f'  F1        (micro): {f1:.3f}')
print()
print(classification_report(y_true, y_pred, scheme=IOB2, digits=3))

## 5. F1 por categoria

In [ ]:
from seqeval.metrics import classification_report as cr
import io as _io

# Parseia o classification_report em DataFrame
report = cr(y_true, y_pred, scheme=IOB2, digits=3, output_dict=True)

rows = []
for ent in ENTITY_NAMES:
    if ent in report:
        d = report[ent]
        rows.append({'Categoria': ent, 'Precision': d['precision'],
                     'Recall': d['recall'], 'F1': d['f1-score'], 'Support': int(d['support'])})
    else:
        rows.append({'Categoria': ent, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'Support': 0})

df_metrics = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
print(df_metrics.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(df_metrics))
w = 0.26

ax.bar(x - w, df_metrics['Precision'], width=w, label='Precision', color='#2980b9', alpha=0.85)
ax.bar(x,     df_metrics['Recall'],    width=w, label='Recall',    color='#27ae60', alpha=0.85)
ax.bar(x + w, df_metrics['F1'],        width=w, label='F1',        color='#e67e22', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_metrics['Categoria'], rotation=25, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 por categoria — Silver vs. Gold')
ax.legend()
for xi, row in zip(x + w, df_metrics.itertuples()):
    ax.text(xi, row.F1 + 0.02, f'{row.F1:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ner_f1_per_category.pdf', bbox_inches='tight')
plt.show()
print(f'Salvo: {FIGURES_DIR / "ner_f1_per_category.pdf"}')

## 6. Matriz de confusão de tipos

Para cada entidade gold, verifica o que o silver predisse na mesma posição de span.
Linha = tipo real (gold), coluna = tipo predito (silver ou `MISS` se ausente).

In [ ]:
def extract_spans(bio_tags):
    """Retorna lista de (tipo, inicio, fim) a partir de BIO tags."""
    spans = []
    i = 0
    while i < len(bio_tags):
        if bio_tags[i].startswith('B-'):
            tipo = bio_tags[i][2:]
            j = i + 1
            while j < len(bio_tags) and bio_tags[j] == f'I-{tipo}':
                j += 1
            spans.append((tipo, i, j))
            i = j
        else:
            i += 1
    return spans


labels_ext = ENTITY_NAMES + ['MISS']
confusion = pd.DataFrame(0, index=ENTITY_NAMES, columns=labels_ext)

for g_seq, p_seq in zip(y_true, y_pred):
    gold_spans = extract_spans(g_seq)
    pred_spans = extract_spans(p_seq)
    pred_idx = {(s, e): t for t, s, e in pred_spans}

    for g_tipo, g_s, g_e in gold_spans:
        pred_tipo = pred_idx.get((g_s, g_e))
        if pred_tipo is None:
            confusion.loc[g_tipo, 'MISS'] += 1
        else:
            confusion.loc[g_tipo, pred_tipo] += 1

# Exibe
print('Matriz de confusão (linhas=gold, colunas=silver):')
print(confusion.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(11, 7))
vals = confusion.values.astype(float)
im = ax.imshow(vals, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xticks(range(len(labels_ext)))
ax.set_yticks(range(len(ENTITY_NAMES)))
ax.set_xticklabels(labels_ext, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ENTITY_NAMES, fontsize=9)
ax.set_xlabel('Silver (predito)')
ax.set_ylabel('Gold (real)')
ax.set_title('Confusão de tipos — Silver vs. Gold')

for i in range(len(ENTITY_NAMES)):
    for j in range(len(labels_ext)):
        v = int(confusion.iloc[i, j])
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                    color='white' if v > vals.max() * 0.55 else 'black')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ner_confusion_types.pdf', bbox_inches='tight')
plt.show()
print(f'Salvo: {FIGURES_DIR / "ner_confusion_types.pdf"}')

## 7. Análise de erros (FP / FN por categoria)

- **FN** (False Negative): entidade no gold que o silver não encontrou ou rotulou errado
- **FP** (False Positive): entidade no silver sem correspondência no gold

In [ ]:
errors = defaultdict(lambda: {'FN': [], 'FP': []})

for sid, g_seq, p_seq in zip(ordered_ids, y_true, y_pred):
    tokens = gold_by_id[sid]['tokens']
    texto = gold_by_id[sid]['texto']
    gold_spans = extract_spans(g_seq)
    pred_spans = extract_spans(p_seq)
    pred_pos = {(s, e): t for t, s, e in pred_spans}
    gold_pos = {(s, e): t for t, s, e in gold_spans}

    for g_tipo, g_s, g_e in gold_spans:
        span_txt = ' '.join(tokens[g_s:g_e])
        p_tipo = pred_pos.get((g_s, g_e))
        if p_tipo != g_tipo:
            errors[g_tipo]['FN'].append({'span': span_txt, 'pred': p_tipo or 'O', 'sent': texto[:80]})

    for p_tipo, p_s, p_e in pred_spans:
        if (p_s, p_e) not in gold_pos:
            span_txt = ' '.join(tokens[p_s:p_e])
            errors[p_tipo]['FP'].append({'span': span_txt, 'sent': texto[:80]})

TOP = 5
for ent in ENTITY_NAMES:
    fn = errors[ent]['FN']
    fp = errors[ent]['FP']
    if not fn and not fp:
        continue
    print(f'\n[{ent}]  FN={len(fn)}  FP={len(fp)}')
    if fn:
        print('  FN (perdidos/errados):')
        for e in fn[:TOP]:
            print(f'    gold="{e["span"]}"  silver_pred={e["pred"]}')
            print(f'    ctx: {e["sent"]}')
    if fp:
        print('  FP (espúrios):')
        for e in fp[:TOP]:
            print(f'    silver="{e["span"]}"')
            print(f'    ctx: {e["sent"]}')

## 8. F1 por relatório (ano)

In [ ]:
from seqeval.metrics import f1_score as f1s

year_rows = []
for ano in sorted({r['relatorio'].replace('sireva_', '') for r in gold_recs}):
    sids = [sid for sid in ordered_ids if gold_by_id[sid]['relatorio'] == f'sireva_{ano}']
    yt = [y_true[ordered_ids.index(s)] for s in sids]
    yp = [y_pred[ordered_ids.index(s)] for s in sids]
    if not yt:
        continue
    f = f1s(yt, yp, scheme=IOB2, average='micro', zero_division=0)
    n_ents = sum(1 for seq in yt for t in seq if t.startswith('B-'))
    year_rows.append({'Ano': ano, 'Sentenças': len(sids), 'Ents gold': n_ents, 'F1': round(f, 3)})

df_year = pd.DataFrame(year_rows)
print(df_year.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_year['Ano'], df_year['F1'], color='#2980b9', edgecolor='white')
ax.axhline(f1, color='red', linestyle='--', label=f'F1 global = {f1:.3f}')
ax.set_ylim(0, 1.05)
ax.set_xlabel('Relatório (ano)')
ax.set_ylabel('F1 (micro)')
ax.set_title('F1 por relatório — Silver vs. Gold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ner_f1_per_year.pdf', bbox_inches='tight')
plt.show()
print(f'Salvo: {FIGURES_DIR / "ner_f1_per_year.pdf"}')

## 9. Tabela LaTeX para o paper

In [ ]:
print('\nTABELA MÉTRICAS POR CATEGORIA (LaTeX)\n')
print('\\begin{table}[h]')
print('\\centering')
print('\\begin{tabular}{lrrrr}')
print('\\toprule')
print('Categoria & Precision & Recall & F1 & Support \\\\')
print('\\midrule')
for _, row in df_metrics.iterrows():
    print(f"{row['Categoria']} & {row['Precision']:.3f} & {row['Recall']:.3f} "
          f"& {row['F1']:.3f} & {int(row['Support'])} \\\\")
print('\\midrule')
print(f'\\textbf{{Micro avg}} & {p:.3f} & {r:.3f} & {f1:.3f} & {n_gold_ents} \\\\')
print('\\bottomrule')
print('\\end{tabular}')
print('\\caption{Avaliação NER: silver (GPT-4o-mini) vs. gold standard (180 sentenças).}')
print('\\label{tab:ner-eval}')
print('\\end{table}')